In [ ]:
# papermill parameters
question = "What is the full path for CPT S 422?"
student_context = {}
use_rag = True

In [ ]:
import sqlite3
import json
import time
import re
import os
from llama_cpp import Llama

# 1. SETUP
db_path = os.path.join(base_dir, "data", "courses.db")
model_path = os.path.join(base_dir, "data", "models", "llama-3.1-8b-q4.gguf")
start_time = time.time()

# Papermill Parameters
question = "What is the full path for CPT S 422?"
student_context = {}
use_rag = True

def fetch_course_info(prefix, number):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT prefix, courseNumber, title, coursePrerequisite 
        FROM courses 
        WHERE REPLACE(UPPER(prefix), ' ', '') LIKE ? AND courseNumber = ? 
        LIMIT 1
    """, (f"%{prefix.upper().replace(' ', '')}%", number))
    row = cursor.fetchone()
    conn.close()
    return row

def build_prereq_chain(query, visited=None):
    """The 'Nervous System': Recursively finds every course in the path."""
    if visited is None: visited = set()
    match = re.search(r'([A-Z]{2,4}(?:\sS)?)\s*(\d{3})', query.upper())
    if not match: return []
    
    prefix, number = match.groups()
    course_id = f"{prefix.strip()} {number}"
    if course_id in visited: return []
    visited.add(course_id)
    
    row = fetch_course_info(prefix, number)
    if not row: return []

    chain = [{"code": f"{row[0]} {row[1]}", "title": row[2], "prereqs": row[3] or "None"}]
    
    # Follow the chain backward
    if row[3] and row[3] != "None":
        found_codes = re.findall(r'([A-Z]{2,4}(?:\sS)?\s*\d{3})', row[3].upper())
        for code in found_codes:
            chain.extend(build_prereq_chain(code, visited))
    return chain

# 2. DECISION LOGIC
# If the user asks for a 'path', 'chain', or 'sequence', use recursion
if any(word in question.lower() for word in ['path', 'chain', 'sequence', 'steps']):
    chain_data = build_prereq_chain(question)
    context_str = "FULL PREREQUISITE CHAIN:\n" + "\n".join([f"- {c['code']}: needs {c['prereqs']}" for c in chain_data])
    sources = [c['code'] for c in chain_data]
else:
    # Standard RAG search for general questions
    context_str = "General advising info..." # (Your existing FAISS logic would go here)
    sources = ["General Catalog"]

# 3. GENERATE ADVICE
llm = Llama(model_path=model_path, n_ctx=4096, verbose=False)
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a WSU advisor. For 'path' questions, explain the sequence from 100-level to target course."},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {question}"}
    ]
)
answer = response['choices'][0]['message']['content'].strip()

# 4. EXPORT
output = {
    "answer": answer,
    "sources": sources,
    "metadata": {
        "used_rag": True, 
        "model": "Llama-3.1-8B", 
        "latency_seconds": round(time.time() - start_time, 2) # Note the key change!
    }
}

temp_file = os.path.join(os.environ.get('TEMP', '/tmp'), 'llm_response.json')
with open(temp_file, 'w') as f:
    json.dump(output, f)
print(json.dumps(output))